# Airspace Object Detection: Unknown-Object Flag (v2)

The combined YOLO11s model knows 6 classes: airplane, bird, helicopter, kite, parachute, drone. Anything else in the sky (a balloon, a blimp, a rocket) it will either miss or force into one of those 6 classes.

This notebook adds a second, **open-vocabulary** detector, YOLO-World, that is prompted with a long list of things that fly. Then it combines the two:

* If YOLO-World finds an object that the known model already detected (boxes overlap), the known model's label wins.
* If YOLO-World finds an object that the known model did **not** detect, it's flagged **UNKNOWN**, and YOLO-World's guess comes along as a hint.

It measures two things:
1. **False unknowns:** how often the flag fires on the normal test images, where every object is a known class. Lower is better.
2. **Unknown catch rate:** on Open Images photos of balloons (never seen in training), what share of balloons get flagged UNKNOWN, and what the known model wrongly calls them.

**How to run:** attach the output of the combined-model notebook (Version 3) as an input, plus the `ROBOFLOW_API_KEY` secret. Use GPU T4 x2 with Internet on, then Save Version → Save & Run All. It takes about 30 to 45 minutes.

**v2 changes (after v1):** v1 flagged too much on normal images, mostly YOLO-World boxes labelled "flag", "kite", "bird" or "airplane", so known objects were counted as unknown. v2 (a) drops the vague prompts "flag", "ball" and "frisbee"; (b) only raises UNKNOWN when YOLO-World's guess is *not* one of the 6 known classes (a known-class guess the main model missed becomes a separate "possible miss"); (c) lets a confident non-known guess **override** the known model, because v1 showed the known model calls many balloons "parachute" or "kite".

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
# 1. Load the known 6-class model (from the combined notebook's output) and YOLO-World
import glob, os
from ultralytics import YOLO
cands = glob.glob("/kaggle/input/**/combined_640/weights/best.pt", recursive=True)
print(cands)
known = YOLO(cands[0])
KNOWN = list(known.names.values())
print("Known classes:", KNOWN)

# Open-vocabulary prompts: the 6 known classes plus other things that can be in the sky
VOCAB = ["airplane", "bird", "helicopter", "kite", "parachute", "drone",
         "balloon", "hot air balloon", "blimp", "airship", "paraglider", "hang glider",
         "glider", "rocket", "missile", "satellite", "flying insect", "butterfly",
         "plastic bag", "sky lantern", "jet", "fighter jet", "quadcopter"]
# YOLO-World guesses that just mean one of the known classes
SYNONYM = {"jet": "airplane", "fighter jet": "airplane", "quadcopter": "drone", "glider": "airplane"}
world = YOLO("yolov8x-worldv2.pt")
world.set_classes(VOCAB)

In [ ]:
# 2. The fusion rule
import numpy as np

def iou(a, b):
    """IoU between boxes a (N,4) and b (M,4), xyxy."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def detect(img, known_conf=0.25, world_conf=0.15, match_iou=0.3, override_margin=0.0):
    """Known-model detections, plus:
       - UNKNOWN for a YOLO-World box whose guess is not a known class and that no known box explains,
       - UNKNOWN override when a known box overlaps a more confident non-known YOLO-World guess,
       - 'possible miss' for a known-class YOLO-World box the main model didn't detect (not counted as unknown)."""
    k = known.predict(img, conf=known_conf, imgsz=640, verbose=False)[0].boxes
    w = world.predict(img, conf=world_conf, imgsz=640, verbose=False)[0].boxes
    kb, wb = k.xyxy.cpu().numpy(), w.xyxy.cpu().numpy()
    out = [dict(box=b, label=KNOWN[int(c)], conf=float(s), unknown=False, miss=False)
           for b, c, s in zip(kb, k.cls.cpu().numpy(), k.conf.cpu().numpy())]
    ov = iou(wb, kb)
    for i, (b, c, s) in enumerate(zip(wb, w.cls.cpu().numpy(), w.conf.cpu().numpy())):
        hint = VOCAB[int(c)]
        is_known = SYNONYM.get(hint, hint) in KNOWN
        if ov.shape[1] == 0 or ov[i].max() < match_iou:
            if is_known:
                out.append(dict(box=b, label=SYNONYM.get(hint, hint), hint=hint, conf=float(s), unknown=False, miss=True))
            else:
                out.append(dict(box=b, label="UNKNOWN", hint=hint, conf=float(s), unknown=True, miss=False))
        elif not is_known:
            j = int(ov[i].argmax())
            if float(s) > out[j]["conf"] + override_margin and not out[j]["unknown"]:
                out[j] = dict(box=b, label="UNKNOWN", hint=hint, conf=float(s), unknown=True, miss=False,
                              was=out[j]["label"])
    return out

In [ ]:
# 3. Known-only test images: both test sets from Roboflow
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
mbaod = rf.workspace("fmk-mwmyi").project("mbaod").version(1).download("yolov8", location="/kaggle/working/mbaod")
aod4  = rf.workspace("airborne-object-detection").project("airborne-object-detection-4-aod4").version(6).download("yolov8", location="/kaggle/working/aod4")

In [ ]:
# 4. False unknowns: run on 1,000 random test images from each set.
#    Every object in these images is one of the known classes, so any UNKNOWN flag is a false alarm.
#    We keep raw YOLO-World scores so we can sweep the threshold afterwards.
import random, collections
random.seed(0)
records = {}   # set name -> list of per-image lists of unknown-flag scores (at world_conf=0.05)
hints = {}
for name, loc in [("MBAOD", mbaod.location), ("AOD 4", aod4.location)]:
    imgs = sorted(glob.glob(f"{loc}/test/images/*"))
    imgs = random.sample(imgs, min(1000, len(imgs)))
    per_img, h = [], collections.Counter()
    for p in imgs:
        dets = detect(p, world_conf=0.05)
        u = [d for d in dets if d["unknown"]]
        per_img.append([d["conf"] for d in u])
        h.update(d["hint"] for d in u if d["conf"] >= 0.15)
    records[name], hints[name] = per_img, h
    print(f"{name}: {len(imgs)} images checked")

THRESH = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
for name, per_img in records.items():
    print(f"\n{name}: share of images with at least one false UNKNOWN flag")
    for t in THRESH:
        rate = np.mean([any(s >= t for s in scores) for scores in per_img])
        print(f"  world_conf {t:.2f}: {rate:.3f}")
    print("  Most common YOLO-World guesses behind false flags (conf>=0.15):", hints[name].most_common(6))

V1_FA = {"MBAOD": [0.381, 0.268, 0.211, 0.161, 0.131, 0.108, 0.064, 0.039],
         "AOD 4": [0.293, 0.200, 0.155, 0.120, 0.093, 0.071, 0.032, 0.021]}
for name, per_img in records.items():
    print(f"\n{name} false-flag rate v1 -> v2:")
    for t, a in zip(THRESH, V1_FA[name]):
        print(f"  world_conf {t:.2f}: {a:.3f} -> {np.mean([any(s >= t for s in sc) for sc in per_img]):.3f}")

In [ ]:
# 5. True unknowns: Open Images photos with balloons (and a few other flying things if present).
#    None of these classes are in the known model's training data.
import pandas as pd, requests, io, concurrent.futures as cf
BASE = "https://storage.googleapis.com/openimages"
cls = pd.read_csv(f"{BASE}/v7/oidv7-class-descriptions-boxable.csv", header=None, names=["mid", "name"])
WANT = ["Balloon", "Rocket", "Missile", "Airship"]
mids = dict(cls[cls.name.isin(WANT)][["mid", "name"]].values)
print("Open Images classes used:", mids)
ann = pd.concat([pd.read_csv(f"{BASE}/v5/{s}-annotations-bbox.csv") for s in ["validation", "test"]])
ann = ann[ann.LabelName.isin(mids) & (ann.IsGroupOf == 0)]
ann = ann[((ann.XMax - ann.XMin) * (ann.YMax - ann.YMin)) > 0.002]   # skip tiny boxes
ids = ann.ImageID.unique()[:400]
os.makedirs("/kaggle/working/unknown_imgs", exist_ok=True)
def fetch(i):
    for split in ["validation", "test"]:
        r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{split}/{i}.jpg", timeout=20)
        if r.ok:
            open(f"/kaggle/working/unknown_imgs/{i}.jpg", "wb").write(r.content); return i
with cf.ThreadPoolExecutor(16) as ex:
    got = [i for i in ex.map(fetch, ids) if i]
ann = ann[ann.ImageID.isin(got)]
print(len(got), "images,", len(ann), "objects:", ann.LabelName.map(mids).value_counts().to_dict())

In [ ]:
# 6. Score the unknowns. For each true unknown object:
#    - caught:      an UNKNOWN flag overlaps it (IoU >= 0.3)
#    - mislabeled:  the known model put a known label on it (e.g. called a balloon a "kite")
#    - missed:      neither
from PIL import Image
rows = []
for i in got:
    path = f"/kaggle/working/unknown_imgs/{i}.jpg"
    W, H = Image.open(path).size
    gt = ann[ann.ImageID == i]
    g = np.stack([gt.XMin * W, gt.YMin * H, gt.XMax * W, gt.YMax * H], 1)
    dets = detect(path, world_conf=0.05)
    ub = np.array([d["box"] for d in dets if d["unknown"]]).reshape(-1, 4)
    us = np.array([d["conf"] for d in dets if d["unknown"]])
    kd = [d for d in dets if not d["unknown"] and not d["miss"]]
    kb = np.array([d["box"] for d in kd]).reshape(-1, 4)
    ou, ok = iou(g, ub), iou(g, kb)
    for j in range(len(g)):
        best_u = us[ou[j] >= 0.3].max() if (ou.shape[1] and (ou[j] >= 0.3).any()) else 0.0
        kl = kd[int(ok[j].argmax())]["label"] if (ok.shape[1] and ok[j].max() >= 0.3) else None
        rows.append(dict(img=i, cls=mids[gt.LabelName.iloc[j]], unk_score=best_u, known_label=kl))
res = pd.DataFrame(rows)
print("Objects:", len(res))
print("\nCatch rate (flagged UNKNOWN) by YOLO-World threshold:")
for t in THRESH:
    print(f"  world_conf {t:.2f}: {(res.unk_score >= t).mean():.3f}")
print("\nWhat the known model called them (when it fired):")
print(res.known_label.fillna("(nothing)").value_counts().to_string())

# v1 for comparison (same images, same thresholds)
V1_CATCH = [0.423, 0.345, 0.312, 0.295, 0.265, 0.242, 0.197, 0.159]
print("\nCatch rate v1 -> v2:")
for t, a in zip(THRESH, V1_CATCH):
    print(f"  world_conf {t:.2f}: {a:.3f} -> {(res.unk_score >= t).mean():.3f}")

In [ ]:
# 7. The trade-off in one chart: false-flag rate vs. catch rate, per threshold
import matplotlib.pyplot as plt, json
fa = {n: [np.mean([any(s >= t for s in sc) for sc in per]) for t in THRESH] for n, per in records.items()}
catch = [(res.unk_score >= t).mean() for t in THRESH]
plt.figure(figsize=(6, 4.5))
for n, v in fa.items():
    plt.plot(v, catch, marker="o", label=f"false flags on {n}")
for t, x, y in zip(THRESH, fa["AOD 4"], catch):
    plt.annotate(f"{t:.2f}", (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
plt.xlabel("Share of known-only images with a false UNKNOWN flag")
plt.ylabel("Share of true unknowns flagged")
plt.title("Unknown-object flag: catch rate vs. false flags")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig("/kaggle/working/unknown_tradeoff.png", dpi=150); plt.show()

summary = {"thresholds": THRESH, "false_flag_rate": fa, "catch_rate": catch,
           "known_label_on_unknowns": res.known_label.fillna("(nothing)").value_counts().to_dict(),
           "n_unknown_objects": len(res), "version": "v2"}
json.dump(summary, open("/kaggle/working/unknown_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

In [ ]:
# 8. Example images: known labels in green, UNKNOWN flags in red (world_conf 0.2)
import cv2
def draw(path, world_conf=0.2):
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    for d in detect(path, world_conf=world_conf):
        x1, y1, x2, y2 = map(int, d["box"])
        col = (255, 40, 40) if d["unknown"] else ((240, 200, 0) if d["miss"] else (40, 200, 40))
        txt = f'UNKNOWN ({d["hint"]}) {d["conf"]:.2f}' if d["unknown"] else f'{d["label"]} {d["conf"]:.2f}'
        cv2.rectangle(im, (x1, y1), (x2, y2), col, 3)
        cv2.putText(im, txt, (x1, max(y1 - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
    return im
ex = random.sample(got, 6) + random.sample(sorted(glob.glob(f"{aod4.location}/test/images/*")), 3)
fig, axs = plt.subplots(3, 3, figsize=(15, 13))
for a, p in zip(axs.flat, ex):
    a.imshow(draw(p if p.startswith("/") else f"/kaggle/working/unknown_imgs/{p}.jpg")); a.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/unknown_examples.jpg", dpi=110); plt.show()

In [ ]:
# 9. Clean up big folders so the Output tab stays small
import shutil
for d in ["/kaggle/working/mbaod", "/kaggle/working/aod4", "/kaggle/working/unknown_imgs"]:
    shutil.rmtree(d, ignore_errors=True)
print(os.listdir("/kaggle/working"))